<font size=10>**PREPROCESSING**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

**Data**: 
- [*Portal BASE*](https://www.base.gov.pt/Base4/pt/pesquisa/?type=contratos&texto=&adjudicante=&adjudicataria=&tipo=2&tipocontrato=0&cpv=&aqinfo=&desdeprazoexecucao=&ateprazoexecucao=&sel_price=price_11&desdeprecocontrato=&ateprecocontrato=&desdeprecoefectivo=&ateprecoefectivo=&sel_date=date_11&desdedatacontrato=2023-01-01&atedatacontrato=2026-03-31&desdedatapublicacao=&atedatapublicacao=&desdedatafecho=&atedatafecho=&pais=0&distrito=0&concelho=0)

- [*Treated Datasets*](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/#/resources)

**Group B**
- Beatriz Marques 20231605
- Maria Inês Santos 20231630
- Luís Soeiro 20211536
- Rodrigo Silva 20231602

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>  
- [1. Imports](#1)  
- [2. Data Integration](#2)  
- [3. Data Preprocessing](#3) 
    - [3.1 Filtering](#31-filtering)
    - [3.2 Drop Data](#32-drop-data)
    - [3.3 Data Types](#33-data-types)
    - [3.4 Text Preprocessing](#34-text-preprocessing)
    - [3.5 Outiers](#35-outiers)
    - [3.6 Missing Values](#36-missing-values)
    - [3.7 Export Preprocessed Data](#37-export-preprocessed-data)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [2]:
import sys
import os

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

In [3]:
import subprocess, sys, importlib
import pandas as pd
import plotly.express as px
import re

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    importlib.invalidate_caches()
    import openpyxl
import os
import shutil

# <font color='#BFD72F' size=6>**2. Data Integration**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

In [4]:
# MERGE DATASETS
paths = {
    "2023_part01": "../data/contratos2023_part01.csv",
    "2023_part02": "../data/contratos2023_part02.csv",
    "2023_part03": "../data/contratos2023_part03.csv",
    "2024_part01": "../data/contratos2024_part01.csv",
    "2024_part02": "../data/contratos2024_part02.csv",
    "2024_part03": "../data/contratos2024_part03.csv",
    "2025_part01": "../data/contratos2025_part01.csv",
    "2025_part02": "../data/contratos2025_part02.csv",
    "2025_part03": "../data/contratos2025_part03.csv",
    "2026": "../data/contratos2026.csv",
}

datasets = {}

merged_dataset = pd.DataFrame()

for year, path in paths.items():
    print(f"Loading dataset for {year} from {path}...")
    datasets[year] = pd.read_csv(path)
    print(f"Dataset for {year} loaded successfully with shape {datasets[year].shape}.")
    merged_dataset = pd.concat([merged_dataset, datasets[year]], ignore_index=True)

print(f"Merged dataset created with shape {merged_dataset.shape}.")

Loading dataset for 2023_part01 from ../data/contratos2023_part01.csv...
Dataset for 2023_part01 loaded successfully with shape (64562, 35).
Loading dataset for 2023_part02 from ../data/contratos2023_part02.csv...
Dataset for 2023_part02 loaded successfully with shape (64562, 35).
Loading dataset for 2023_part03 from ../data/contratos2023_part03.csv...
Dataset for 2023_part03 loaded successfully with shape (64562, 35).
Loading dataset for 2024_part01 from ../data/contratos2024_part01.csv...
Dataset for 2024_part01 loaded successfully with shape (75440, 35).
Loading dataset for 2024_part02 from ../data/contratos2024_part02.csv...
Dataset for 2024_part02 loaded successfully with shape (75440, 35).
Loading dataset for 2024_part03 from ../data/contratos2024_part03.csv...
Dataset for 2024_part03 loaded successfully with shape (75441, 35).
Loading dataset for 2025_part01 from ../data/contratos2025_part01.csv...
Dataset for 2025_part01 loaded successfully with shape (81131, 35).
Loading datas

# <font color='#BFD72F' size=6>**3. Data Preprocessing**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.1 Filtering</font> <a class="anchor" id="3.1"></a>
  
[Back to TOC](#toc)

**Filters Applied**: 
- Procurement Procedure Type: Public
- Contract Start Date: 2023-01-01   
- Contract End Date: 2026-04-25 

In [5]:
subset = merged_dataset[merged_dataset['tipoprocedimento'] == 'Concurso público']

cols_to_keep = [
    'idcontrato', 'tipoContrato', 'tipoFimContrato', 'CPV', 
    'adjudicante', 'adjudicatarios', 'concorrentes', 
    'precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo', 
    'LocalExecucao', 
    "dataDecisaoAdjudicacao", "dataCelebracaoContrato", "dataPublicacao", "dataFechoContrato"
]

subset = subset[cols_to_keep]

In [6]:
subset.shape

(113185, 15)

In [7]:
print("There are {} Public Entities.".format(subset['adjudicante'].nunique()))
print("There are {} Companies.".format(subset['adjudicatarios'].nunique()))

print("So, in total our analysis contains {} Nodes.".format(
    subset['adjudicatarios'].nunique() + 
    subset['adjudicante'].nunique()))

There are 4015 Public Entities.
There are 26110 Companies.
So, in total our analysis contains 30125 Nodes.


In [8]:
total_contracts = subset.shape[0]
print("{} Contracts, characterised by {} columns.".format(total_contracts, subset.shape[1]))

113185 Contracts, characterised by 15 columns.


## <font color='#BFD72F' size=6>3.2 Drop Data</font> <a class="anchor" id="3.2"></a>
  
[Back to TOC](#toc)

In [9]:
# duplicated rows
subset = subset.drop_duplicates()

In [10]:
# rows with missing values except for 'concorrentes', 'dataFechoContrato' and 'tipoFimContrato'
subset = subset.dropna(subset=[col for col in subset.columns if col not in ['concorrentes', 'dataFechoContrato', 'tipoFimContrato']])

In [11]:
print("Initial Number of Contracts: {}".format(total_contracts))
print("Number of Contracts Now: {}".format(subset.shape[0]))
print("Percentage of Deleted Contracts: {}%".format(round((1 - (subset.shape[0]/total_contracts))*100, 2)))

Initial Number of Contracts: 113185
Number of Contracts Now: 112170
Percentage of Deleted Contracts: 0.9%


## <font color='#BFD72F' size=6>3.3 Data Types</font> <a class="anchor" id="3.3"></a>
  
[Back to TOC](#toc)

In [12]:
# TODO: date columns     
subset["dataPublicacao"] = pd.to_datetime(subset["dataPublicacao"], errors='coerce')
subset["dataCelebracaoContrato"] = pd.to_datetime(subset["dataCelebracaoContrato"], errors='coerce')
subset["dataDecisaoAdjudicacao"] = pd.to_datetime(subset["dataDecisaoAdjudicacao"], errors='coerce')
subset["dataFechoContrato"] = pd.to_datetime(subset["dataFechoContrato"], errors='coerce')

## <font color='#BFD72F' size=6>3.4 Text Preprocessing</font> <a class="anchor" id="3.4"></a>
  
[Back to TOC](#toc)

In [13]:
# tipoContrato
subset['tipoContrato'] = subset['tipoContrato'].astype(str).str.replace(r'[\r\n]+', ' | ', regex=True).str.strip()

In [14]:
# CPV
subset['CPV'] = subset['CPV'].astype(str).str.replace('\n', ' | ', regex=True).str.strip()
subset['CPV'] = subset['CPV'].astype(str).str.replace(r'\s*\d{8}-\d\s*-\s*', ' ', regex=True).str.strip()

In [15]:
# concorrentes
subset['concorrentes'] = (
    subset['concorrentes']
    .astype(str)
    # replace line breaks with separator
    .str.replace(r'[\r\n]+', ' | ', regex=True)
    # remove excessive spaces
    .str.replace(r'\s+', ' ', regex=True)
    # remove trailing numbers (like " 102", " 89", etc.)
    .str.replace(r'\s+\d+\s*$', '', regex=True)
    # final trim
    .str.strip()
)

In [16]:
# LocalExecucao
subset['LocalExecucao'] = subset['LocalExecucao'].fillna('')

subset['LocalExecucao'] = (
    subset['LocalExecucao']
    # replace line breaks with separator
    .str.replace(r'[\r\n]+', ' | ', regex=True)
    # normalize spaces
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    # remove duplicates inside each cell
    .apply(lambda x: ' | '.join(dict.fromkeys(x.split(' | '))) if x else x)
)

first_location = subset['LocalExecucao'].str.split(' \| ', expand=False).str[0]

split_cols = first_location.str.split(', ', expand=True)

split_cols = split_cols.reindex(columns=[0, 1, 2])
split_cols.columns = ['country', 'district', 'city']

subset[['country', 'district', 'city']] = split_cols

for col in ['country', 'district', 'city']:
    subset[col] = subset[col].replace(r'^\s*$', pd.NA, regex=True)

# handle inconsistent structures
n_parts = first_location.str.split(', ').str.len()

# if only 1 part - it's country
subset.loc[n_parts == 1, ['district', 'city']] = pd.NA

# if 2 parts - assume country + district
subset.loc[n_parts == 2, 'city'] = pd.NA

subset = subset[subset['country'] == 'Portugal']
subset = subset.drop(columns=['LocalExecucao', 'country'])

In [17]:
print("Initial Number of Contracts: {}".format(total_contracts))
print("Number of Contracts Now: {}".format(subset.shape[0]))
print("Percentage of Deleted Contracts: {}%".format(round((1 - (subset.shape[0]/total_contracts))*100, 2)))

Initial Number of Contracts: 113185
Number of Contracts Now: 112103
Percentage of Deleted Contracts: 0.96%


In [18]:
# extract contribuinte numbers from adjudicante and adjudicatarios
subset['contribuinte_adjudicante'] = subset['adjudicante'].str.extract(r'(\d{9})')
subset['adjudicante'] = subset['adjudicante'].str.replace(r'\s*\d{9}\s* - ', '', regex=True).str.strip()

subset['contribuinte_adjudicatarios'] = subset['adjudicatarios'].str.extract(r'(\d{9})')
subset['adjudicatarios'] = subset['adjudicatarios'].str.replace(r'\s*\d{9}\s* - ', '', regex=True).str.strip()

In [19]:
# number of competitors per contract
subset['nr_concorrentes'] = subset['concorrentes'].apply(
    lambda x: len([i for i in re.split(r'\s*\|\s*', x) if i]) 
    if isinstance(x, str) else 0
)

fig = px.histogram(
    subset,
    x='nr_concorrentes',
    nbins=30,
    title='Histogram of Number of Competitors'
)

fig.show()

In [20]:
subset.head()

,idcontrato,tipoContrato,tipoFimContrato,CPV,adjudicante,adjudicatarios,concorrentes,precoBaseProcedimento,precoContratual,PrecoTotalEfetivo,dataDecisaoAdjudicacao,dataCelebracaoContrato,dataPublicacao,dataFechoContrato,district,city,contribuinte_adjudicante,contribuinte_adjudicatarios,nr_concorrentes
2,9664960,Aquisição de bens móveis,Anulado ou Declarado Nulo,Computadores portáteis,Universidade do Algarve,"Empis - Informática e Serviços, Lda",501333401-BASE2 - Informática e Telecomunicaçõ...,320620.41,3260.00,0.00,2022-12-12,2023-01-02,2023-01-02,2023-12-31,Faro,Faro,505387271,502163518,15
15,9667338,Aquisição de serviços,NaN,Serviços de viagens,Direção-Geral da Administração da Justiça,"Dot Viagens e Turismo, Lda",506019608-Smile Viagens e Turismo Unipessoal L...,81895.31,81895.31,0.00,2022-12-15,2023-01-03,2023-01-03,NaT,<NA>,<NA>,600072525,514862645,13
24,9668634,Aquisição de bens móveis,Resolução do Contrato,Gasóleo,Município de Góis,Alves Bandeira SA,500433402-Alves Bandeira SA,374112.00,327326.40,343994.23,2022-12-07,2023-01-02,2023-01-03,2026-01-16,Coimbra,Góis,506613399,500433402,1
27,9671613,Empreitadas de obras públicas,NaN,Obras de revisão e recuperação,Município de Braga,"DAMOS COR, LDA.","513129596-DAMOS COR, LDA. | 500553408-Alexandr...",318935.89,318343.79,0.00,2022-12-09,2023-01-03,2023-01-04,NaT,Braga,Braga,506901173,513129596,2
30,9664045,Aquisição de bens móveis,"O cumprimento, a impossibilidade definitiva e ...",Produtos de panificação,Serviços de Ação Social da Universidade do Minho,Padaria Trinas lda,500209634-Padaria Trinas lda,85424.72,84659.05,84659.05,2022-12-27,2023-01-01,2023-01-02,2023-08-22,Braga,Braga,680047360,500209634,1


## <font color='#BFD72F' size=6>3.5 Outiers</font> <a class="anchor" id="3.5"></a>
  
[Back to TOC](#toc)

## <font color='#BFD72F' size=6>3.6 Missing Values</font> <a class="anchor" id="3.6"></a>
  
[Back to TOC](#toc)

In [21]:
total = len(subset)
missing_counts = subset.isnull().sum()
missing_pct = (missing_counts / total * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

# show only columns with any missing values
missing_df

,missing_count,missing_pct
dataFechoContrato,95106,84.84
tipoFimContrato,95064,84.80
concorrentes,28044,25.02
city,18763,16.74
district,16103,14.36
contribuinte_adjudicatarios,3308,2.95
contribuinte_adjudicante,251,0.22
tipoContrato,0,0.00
idcontrato,0,0.00
adjudicante,0,0.00


## <font color='#BFD72F' size=6>3.7 Export Preprocessed Data</font> <a class="anchor" id="3.7"></a>
  
[Back to TOC](#toc)

In [22]:
subset.to_csv("../data/preprocessed_data.csv", index=False)